In [4]:
import pandas as pd

In [6]:
path = "../data/df_diabetic.csv"

df = pd.read_csv(path, low_memory=False)

df.head()

,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),6,25,1,1,NaN,...,No,No,No,No,No,No,No,No,No,0
1,149190,55629189,Caucasian,Female,[10-20),1,1,7,3,NaN,...,No,Up,No,No,No,No,No,Ch,Yes,0
2,64410,86047875,AfricanAmerican,Female,[20-30),1,1,7,2,NaN,...,No,No,No,No,No,No,No,No,Yes,0
3,500364,82442376,Caucasian,Male,[30-40),1,1,7,2,NaN,...,No,Up,No,No,No,No,No,Ch,Yes,0
4,16680,42519267,Caucasian,Male,[40-50),1,1,7,1,NaN,...,No,Steady,No,No,No,No,No,Ch,Yes,0


## Splitting Data

In [16]:
from sklearn.model_selection import train_test_split

X = df.drop("readmitted", axis=1)
y = df["readmitted"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## Transforming

In [17]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy="median")),
        ('std_scaler', StandardScaler()),
    ])

In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

num_attribs = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_attribs = X_train.select_dtypes(include=["object"]).columns.tolist()

full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_attribs),
])

X_train_prepared = full_pipeline.fit_transform(X_train)
X_test_prepared = full_pipeline.transform(X_test)

## Models

In [50]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, accuracy_score

In [56]:
def model_results(y_test, y_pred, name):
    print(f"=={name}==\n")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("\nPrecision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("\nF1 Score:", f1_score(y_test, y_pred))
    print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [57]:
model = LogisticRegression(max_iter=1000)

model.fit(X_train_prepared, y_train)

y_pred = model.predict(X_test_prepared)

model_results(y_test, y_pred, "Logistic Regression")

==Logistic Regression==

Accuracy: 0.888375749238479

Precision: 0.4945054945054945
Recall: 0.019815059445178335

F1 Score: 0.03810330228619814

Confusion Matrix:
 [[18037    46]
 [ 2226    45]]


In [58]:
model = RandomForestClassifier(n_estimators=100, random_state=631)

model.fit(X_train_prepared, y_train)

y_pred = model.predict(X_test_prepared)

model_results(y_test, y_pred, "Random Forest Classifier")

==Random Forest Classifier==

Accuracy: 0.8890144443352658

Precision: 0.7727272727272727
Recall: 0.007485689123734038

F1 Score: 0.014827736589620584

Confusion Matrix:
 [[18078     5]
 [ 2254    17]]
